# บทที่ 06 · จำลองคำสั่ง เงินสด และหุ้นทีละเหตุการณ์

เรียนรู้ target/order/fill, partial fill, การยกเลิกส่วนค้าง และการตรวจ cash/position ด้วยข้อมูล SYNTHETIC 7 session

ก่อนเริ่ม: อ่าน Python class และรู้การคิดต้นทุน ใช้ Python 3.12 และ standard library เท่านั้น ข้อมูลทุกแถวอยู่ในไฟล์ ไม่มีเครือข่ายหรือบัญชีจริง

เงินเริ่มต้น 1,000 USD หุ้นเต็มหน่วย ไม่มีชอร์ต/leverage/corporate actions ไม่มีวันที่หรือ timezone จริง ราคาซื้อแพงกว่าเปิด 0.1% ราคาขายต่ำกว่าเปิด 0.1% ปัดเป็นเซนต์ half-up ค่าคงที่ 1 USD ต่อ fill capacity เป็นข้อจำกัดที่สร้างขึ้น ไม่ใช่ order book จริง

## 1. อ่านลำดับข้อมูล

Target ในแต่ละแถวเพิ่งพร้อมหลังปิด session นั้น คำสั่งจึงเริ่ม fill ได้ตั้งแต่เปิด session ถัดไป ส่วน capacity คือจำนวนหุ้นสูงสุดที่ยอมให้คำสั่งของเราจับคู่ ณ ราคาเปิดนั้น

In [1]:
from dataclasses import dataclass
from decimal import Decimal, ROUND_FLOOR, ROUND_HALF_UP
import platform

D = Decimal
print({"python": platform.python_version(), "arithmetic": "decimal; prices rounded half-up to cents"})
bars = [
    # session, open, close, simulated available shares at this open, close target
    (1, "100", "101", 10, 8),
    (2, "101", "103", 3, 8),
    (3, "103", "99", 3, 0),
    (4, "99", "98", 10, 0),
    (5, "98", "102", 10, 5),
    (6, "102", "104", 2, 0),
    (7, "103", "103", 10, 0),
]
print("session | open | close | capacity | target_after_close")
for bar in bars:
    print(" | ".join(map(str, bar)))


{'python': '3.12.14', 'arithmetic': 'decimal; prices rounded half-up to cents'}
session | open | close | capacity | target_after_close
1 | 100 | 101 | 10 | 8
2 | 101 | 103 | 3 | 8
3 | 103 | 99 | 3 | 0
4 | 99 | 98 | 10 | 0
5 | 98 | 102 | 10 | 5
6 | 102 | 104 | 2 | 0
7 | 103 | 103 | 10 | 0


## 2. เก็บสถานะให้ครบ

Simulator มีเงินสด หุ้นจริง pending order และ log แยก orders/fills

set_target ไม่ส่งคำสั่งซ้ำหากหุ้นจริงบวกยอดค้างตรงเป้าหมายอยู่แล้ว หาก target เปลี่ยน จะยกเลิกยอดค้างเดิมก่อนคำนวณ delta ใหม่ การ cancel ในแบบจำลองสำเร็จทันที ระบบจริงต้องรอยืนยัน

on_open ตรวจเวลาและเงินสดก่อน fill ใช้ Decimal ที่สร้างจาก string เพื่อคุมเลขฐานสิบ ค่าธรรมเนียมนี้เป็นสมมติฐานของบทเรียน ไม่ใช่ Webull

In [2]:
@dataclass
class Order:
    order_id: str
    side: int
    remaining: int
    submitted_session: int


class Simulator:
    """Single asset, integer shares, at most one active order, no leverage."""
    def __init__(self, cash="1000", fee="1", slippage="0.001"):
        self.cash, self.fee, self.slippage = D(cash), D(fee), D(slippage)
        self.position = 0
        self.pending = None
        self.events, self.fills, self.ledger = [], [], []
        self.sequence = 0

    def set_target(self, session, target):
        if not isinstance(target, int) or target < 0:
            raise ValueError("Target must be a nonnegative integer")
        delta = target - self.position
        side = 1 if delta > 0 else -1
        if self.pending and self.pending.side == side and self.pending.remaining == abs(delta):
            return
        if self.pending:
            self.events.append((session, self.pending.order_id, "CANCELLED", self.pending.remaining))
            self.pending = None
        if delta:
            self.sequence += 1
            self.pending = Order(f"O{self.sequence:02d}", side, abs(delta), session)
            self.events.append((session, self.pending.order_id, "SUBMITTED", abs(delta)))

    def on_open(self, session, open_price, capacity):
        if not self.pending:
            return
        if session <= self.pending.submitted_session:
            raise ValueError("An order cannot fill before its next session")
        if capacity < 0 or int(capacity) != capacity:
            raise ValueError("Capacity must be a nonnegative integer")
        order = self.pending
        price = (D(open_price) * (1 + order.side * self.slippage)).quantize(D(".01"), rounding=ROUND_HALF_UP)
        if price <= 0:
            raise ValueError("Price must be positive")
        quantity = min(order.remaining, capacity)
        if not quantity:
            return
        if order.side == 1:
            affordable = max(0, int(((self.cash - self.fee) / price).to_integral_value(rounding=ROUND_FLOOR)))
            quantity = min(quantity, affordable)
            if quantity == 0:
                self.events.append((session, order.order_id, "REJECTED_CASH", order.remaining))
                self.pending = None
                return
        else:
            quantity = min(quantity, self.position)
            if quantity == 0 or self.cash + quantity * price < self.fee:
                self.events.append((session, order.order_id, "REJECTED_SELL", order.remaining))
                self.pending = None
                return
        self.cash -= order.side * quantity * price + self.fee
        self.position += order.side * quantity
        order.remaining -= quantity
        state = "FILLED" if order.remaining == 0 else "PARTIALLY_FILLED"
        self.events.append((session, order.order_id, state, order.remaining))
        self.fills.append({"session": session, "order": order.order_id, "side": "BUY" if order.side == 1 else "SELL", "quantity": quantity, "fill_price": price, "fee": self.fee, "cash_after": self.cash, "position_after": self.position})
        if not order.remaining:
            self.pending = None
        assert self.cash >= 0 and self.position >= 0

    def mark_close(self, session, close_price):
        equity = self.cash + self.position * D(close_price)
        self.ledger.append({"session": session, "cash": self.cash, "position": self.position, "close": D(close_price), "equity": equity})


## 3. เดินทีละเหตุการณ์

แต่ละ session ต้อง open → fills → mark close → new target ตามลำดับ ไม่ใช้ราคาปิดสร้างสัญญาณแล้วย้อนมา fill ที่เปิดแท่งเดิม

In [3]:
sim = Simulator()
for session, opening, closing, capacity, target in bars:
    sim.on_open(session, opening, capacity)
    sim.mark_close(session, closing)
    sim.set_target(session, target)
print("session | cash | shares | equity at close")
for row in sim.ledger:
    print(row["session"], row["cash"], row["position"], row["equity"], sep=" | ")


session | cash | shares | equity at close
1 | 1000 | 0 | 1000
2 | 695.70 | 3 | 1004.70
3 | 385.40 | 6 | 979.40
4 | 977.80 | 0 | 977.80
5 | 977.80 | 0 | 977.80
6 | 772.60 | 2 | 980.60
7 | 977.40 | 0 | 977.40


## 4. อ่าน order log และ fill log แยกกัน

คำสั่ง O01 ขอ 8 หุ้น ได้ 3 แล้วอีก 3 จากนั้นยกเลิกส่วนที่เหลือ 2 หุ้น การ fill สองรอบคิดค่าคงที่สองครั้ง จบตัวอย่างด้วยเงิน 977.40 USD และหุ้น 0

In [4]:
print("session | order | state | remaining")
for event in sim.events:
    print(*event, sep=" | ")
print("session | side | shares | fill price | fee | cash after")
for fill in sim.fills:
    print(fill["session"], fill["side"], fill["quantity"], fill["fill_price"], fill["fee"], fill["cash_after"], sep=" | ")
print({"final_cash_usd": str(sim.cash), "net_pnl_usd": str(sim.cash - D("1000")), "fill_count": len(sim.fills), "fees_usd": str(sum(fill["fee"] for fill in sim.fills))})


session | order | state | remaining
1 | O01 | SUBMITTED | 8
2 | O01 | PARTIALLY_FILLED | 5
3 | O01 | PARTIALLY_FILLED | 2
3 | O01 | CANCELLED | 2
3 | O02 | SUBMITTED | 6
4 | O02 | FILLED | 0
5 | O03 | SUBMITTED | 5
6 | O03 | PARTIALLY_FILLED | 3
6 | O03 | CANCELLED | 3
6 | O04 | SUBMITTED | 2
7 | O04 | FILLED | 0
session | side | shares | fill price | fee | cash after
2 | BUY | 3 | 101.10 | 1 | 695.70
3 | BUY | 3 | 103.10 | 1 | 385.40
4 | SELL | 6 | 98.90 | 1 | 977.80
6 | BUY | 2 | 102.10 | 1 | 772.60
7 | SELL | 2 | 102.90 | 1 | 977.40
{'final_cash_usd': '977.40', 'net_pnl_usd': '-22.60', 'fill_count': 5, 'fees_usd': '5'}


## 5. พิสูจน์สมดุลบัญชี

สร้างเงินสดใหม่จาก fill log ไม่พึ่งยอด cash ที่ simulator สรุป ตรวจ equity = cash + shares × close ทุกแถว และเทียบกำไรขาดทุนราคาอ้างอิง -16 USD ลบ slippage 1.60 USD กับค่าคงที่ 5 USD

In [5]:
cash = D("1000")
shares = 0
for fill in sim.fills:
    sign = 1 if fill["side"] == "BUY" else -1
    cash = cash - sign * fill["quantity"] * fill["fill_price"] - fill["fee"]
    shares += sign * fill["quantity"]
    assert cash == fill["cash_after"] and shares == fill["position_after"]
assert cash == D("977.40") and shares == 0 and sim.pending is None
assert len(sim.fills) == 5
assert len([event for event in sim.events if event[2] == "CANCELLED"]) == 2
assert sim.fills[0]["session"] == 2 and sim.fills[0]["quantity"] == 3
for row in sim.ledger:
    assert row["equity"] == row["cash"] + row["position"] * row["close"]
# Arithmetic independent of the event loop: reference-open P&L less slippage and fees.
reference_pnl = 6 * D("99") - 3 * D("101") - 3 * D("103") + 2 * (D("103") - D("102"))
slippage_cost = sum(D(fill["quantity"]) * abs(fill["fill_price"] - D(bars[fill["session"] - 1][1])) for fill in sim.fills)
assert D("1000") + reference_pnl - slippage_cost - D("5") == cash
print({"reference_open_pnl_usd": str(reference_pnl), "slippage_cost_usd": str(slippage_cost), "accounting": "PASS"})


{'reference_open_pnl_usd': '-16', 'slippage_cost_usd': '1.60', 'accounting': 'PASS'}


## 6. ลองเหตุการณ์ที่ไม่สำเร็จ

ทำนายก่อนรัน: เงิน 100 USD ซื้อหุ้นที่ราคา 101.10 พร้อม fee 1 ได้หรือไม่? เป้าหมายเดิม 8 แต่ถือ 3 ค้าง 5 ต้องส่งซ้ำหรือไม่? capacity 0 ทำให้เงินเปลี่ยนหรือไม่? เซลล์นี้ตรวจทั้งสามกรณีและกรณี fill ก่อนเวลา

In [6]:
poor = Simulator(cash="100")
poor.set_target(1, 8)
poor.on_open(2, "101", 10)
assert poor.cash == D("100") and poor.position == 0
assert poor.events[-1][2] == "REJECTED_CASH" and not poor.fills
duplicate = Simulator()
duplicate.set_target(1, 8)
duplicate.on_open(2, "101", 3)
duplicate.set_target(2, 8)
assert duplicate.sequence == 1 and duplicate.pending.remaining == 5
assert duplicate.position == 3
duplicate.on_open(3, "103", 0)
assert duplicate.position == 3 and duplicate.pending.remaining == 5
try:
    too_early = Simulator()
    too_early.set_target(2, 8)
    too_early.on_open(2, "101", 10)
except ValueError:
    pass
else:
    raise AssertionError("Same-session fill should fail")
print("PASS: insufficient cash, repeated target, zero liquidity, and same-session rejection")


PASS: insufficient cash, repeated target, zero liquidity, and same-session rejection


## แบบฝึกหัดและเฉลย

1. หลัง close session 2 ถือ 3 ค้างซื้อ 5 เป้าหมาย 8: ไม่สร้าง order ใหม่
2. capacity 0: ไม่เกิด fill เงินสด หุ้น และ remaining คงเดิม
3. หลัง close session 3 ถือ 6 ค้างซื้อ 2 แต่ target 0: cancel ส่วน 2 แล้วส่ง sell 6 ไม่ใช่ sell 8

เลือกเพิ่มกรณีหนึ่ง เช่นค่าคงที่ 2 USD ต่อ fill แล้วคำนวณ ledger ใหม่ ต้องไม่สรุปจากการลบส่วนต่าง fee อย่างเดียวหากเงินที่เหลืออาจเปลี่ยนจำนวนหุ้นที่ซื้อไหว

แบบจำลองยังไม่รวมคิวคำสั่ง latency trading halt settlement taxes หรือ cancel/fill race จึงใช้ตรวจ logic ไม่ใช่หลักฐาน fill ที่จะได้จากตลาดจริง

## แหล่งอ้างอิงและสถานะการรัน

Yves Hilpisch, Python for Algorithmic Trading; https://github.com/yhilpisch/py4at; บท 6 หน้า 175–178 (PDF 195–198)

- https://docs.python.org/3/library/decimal.html

ตรวจแหล่งออนไลน์ 11 กันยายน 2026 บทเรียน ข้อมูล และโค้ดเขียนใหม่ ไม่ได้แนบหนังสือหรือแจกโค้ดต้นฉบับของ Hilpisch

มีผลจากการรัน Python ทุกเซลล์ตามลำดับจาก state ว่างแนบไว้แล้ว ไม่มีการเชื่อมเครือข่าย กด Restart Kernel แล้ว Run All เพื่อทำซ้ำใน Jupyter ได้